# Post-Analysis: Fama-French Alpha, Shock Events & Failure Modes

**Prerequisites:** Run after v3.0 completes. Needs saved model states from v3.0 output.

**Contents:**
1. Fama-French 3-factor market-neutral alpha
2. Shock event attention analysis (COVID crash, 2022 rate hikes)
3. Failure mode documentation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
V3_DIR = "/content/drive/MyDrive/2026 Spring/STAT 3106/3106_Projects/Projects/output/v_0416_v3.0"
OUT_DIR = "/content/drive/MyDrive/2026 Spring/STAT 3106/3106_Projects/Projects/output/v_post_analysis"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"v3.0 output: {V3_DIR}")
print(f"Post-analysis output: {OUT_DIR}")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.sparse.csgraph import laplacian
from scipy.linalg import eigh
from scipy.stats import spearmanr, rankdata
from scipy.sparse import csr_matrix
import json, time, warnings, io
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 0. Rebuild Data Pipeline + Load Models

We need the full data pipeline to generate predictions. This replicates v3.0 cells 4-14.

In [ ]:
# ── Data loading (same as v3.0) ──
DATA = "/content/drive/MyDrive/2026 Spring/STAT 3106/3106_Projects/Projects/Data"
crsp = pd.read_csv(f"{DATA}/qf0egyr4ffi0pszj.csv", parse_dates=["DlyCalDt"])
crsp = crsp.sort_values(["PERMNO", "DlyCalDt"]).reset_index(drop=True)
compustat = pd.read_csv(f"{DATA}/gg3axrtvut5hi5hh.csv", parse_dates=["datadate"])
compustat = compustat.rename(columns={"LPERMNO": "PERMNO"})

N_TARGET = 300
date_range = crsp["DlyCalDt"].nunique()
stock_counts = crsp.groupby("PERMNO")["DlyCalDt"].count()
valid_permnos = stock_counts[stock_counts >= date_range * 0.8].index
gsector_map = compustat.drop_duplicates("PERMNO", keep="last").set_index("PERMNO")["gsector"]
valid_permnos = valid_permnos[valid_permnos.isin(gsector_map.index)]
avg_vol = crsp[crsp["PERMNO"].isin(valid_permnos)].groupby("PERMNO")["DlyVol"].mean()
top_permnos = sorted(avg_vol.nlargest(N_TARGET).index.tolist())
crsp_sub = crsp[crsp["PERMNO"].isin(top_permnos)].copy()
permno_to_idx = {p: i for i, p in enumerate(top_permnos)}
N_STOCKS = len(top_permnos)
sectors = np.array([gsector_map.get(p, -1) for p in top_permnos])
print(f"Selected {N_STOCKS} stocks")

In [ ]:
print("Computing technical indicators...")
t0 = time.time()

def add_technicals(df):
    results = []
    total = df["PERMNO"].nunique()
    for i, (permno, g) in enumerate(df.groupby("PERMNO")):
        if (i + 1) % 100 == 0: print(f"  {i+1}/{total} stocks...")
        g = g.sort_values("DlyCalDt").copy()
        c, h, l, v, r = g["DlyPrc"], g["DlyHigh"], g["DlyLow"], g["DlyVol"], g["DlyRet"]
        cs = c.replace(0, np.nan)
        delta = c.diff()
        avg_gain = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
        avg_loss = (-delta).clip(lower=0).ewm(span=14, adjust=False).mean()
        g["RSI"] = (100 - 100 / (1 + avg_gain / avg_loss.replace(0, np.nan))) / 100
        ema12 = c.ewm(span=12, adjust=False).mean()
        ema26 = c.ewm(span=26, adjust=False).mean()
        macd = ema12 - ema26
        g["MACD_hist"] = (macd - macd.ewm(span=9, adjust=False).mean()) / cs
        sma20 = c.rolling(20, min_periods=10).mean()
        std20 = c.rolling(20, min_periods=10).std()
        g["BB_pctB"] = (c - (sma20 - 2 * std20)) / (4 * std20).replace(0, np.nan)
        tr = pd.concat([h - l, (h - c.shift(1)).abs(), (l - c.shift(1)).abs()], axis=1).max(axis=1)
        g["ATR_norm"] = tr.rolling(14, min_periods=7).mean() / cs
        for w in [5, 20, 60]:
            sma = c.rolling(w, min_periods=max(1, w // 2)).mean()
            g[f"close_sma{w}"] = c / sma.replace(0, np.nan)
        for w in [5, 20]: g[f"mom_{w}d"] = c.pct_change(w)
        for w in [5, 20]: g[f"rvol_{w}d"] = r.rolling(w, min_periods=max(1, w // 2)).std() * np.sqrt(252)
        g["vol_ratio"] = v / v.rolling(20, min_periods=10).mean().replace(0, np.nan)
        g["log_vol_chg"] = np.log1p(v).diff()
        g["ba_spread"] = (g["DlyAsk"] - g["DlyBid"]) / cs
        g["turnover"] = v / (g["ShrOut"].replace(0, np.nan) * 1000)
        g["excess_ret"] = r - g["sprtrn"]
        results.append(g)
    return pd.concat(results, ignore_index=True)

crsp_sub = add_technicals(crsp_sub)
print(f"Done in {time.time() - t0:.1f}s")

In [ ]:
TECH_COLS = ["RSI", "MACD_hist", "BB_pctB", "ATR_norm",
             "close_sma5", "close_sma20", "close_sma60",
             "mom_5d", "mom_20d", "rvol_5d", "rvol_20d",
             "vol_ratio", "log_vol_chg", "ba_spread", "turnover", "excess_ret"]
FUND_COLS = ["roe", "leverage", "profit_margin", "log_assets"]

def build_feature_tensor(df, permnos, p2i):
    dates = sorted(df["DlyCalDt"].unique())
    T, N = len(dates), len(permnos)
    F = 6 + len(TECH_COLS)
    feat = np.full((T, N, F), np.nan, dtype=np.float32)
    d2t = {d: t for t, d in enumerate(dates)}
    df = df.copy()
    df["_t"] = df["DlyCalDt"].map(d2t)
    df["_n"] = df["PERMNO"].map(p2i)
    df = df.dropna(subset=["_t", "_n"])
    t = df["_t"].astype(int).values; n = df["_n"].astype(int).values
    c = df["DlyPrc"].values.astype(np.float32)
    cs = np.where(c == 0, 1.0, c)
    feat[t, n, 0] = c
    feat[t, n, 1] = df["DlyOpen"].values / cs
    feat[t, n, 2] = df["DlyHigh"].values / cs
    feat[t, n, 3] = df["DlyLow"].values / cs
    feat[t, n, 4] = np.log1p(df["DlyVol"].values)
    feat[t, n, 5] = df["DlyRet"].values
    for i, col in enumerate(TECH_COLS):
        feat[t, n, 6 + i] = df[col].values
    return feat, dates

features, dates = build_feature_tensor(crsp_sub, top_permnos, permno_to_idx)

fund = compustat[["PERMNO", "datadate", "rdq", "atq", "ceqq", "niq"]].copy()
fund["rdq"] = pd.to_datetime(fund["rdq"])
fund["avail_date"] = fund["rdq"].fillna(fund["datadate"] + pd.Timedelta(days=45))
def safe_ratio(num, denom, clip=10):
    return (num / denom.replace(0, np.nan)).clip(-clip, clip)
fund["roe"] = safe_ratio(fund["niq"], fund["ceqq"])
fund["leverage"] = safe_ratio(fund["atq"], fund["ceqq"])
fund["profit_margin"] = safe_ratio(fund["niq"], fund["atq"])
fund["log_assets"] = np.log1p(fund["atq"].clip(lower=0))

T, N, F_base = features.shape
fund_feat = np.full((T, N, len(FUND_COLS)), np.nan, dtype=np.float32)
dates_arr = np.array(dates)
for permno in top_permnos:
    ni = permno_to_idx[permno]
    sf = fund[fund["PERMNO"] == permno].sort_values("avail_date")
    for _, row in sf.iterrows():
        mask = dates_arr >= row["avail_date"]
        if mask.any():
            fund_feat[mask, ni, :] = [row[c] for c in FUND_COLS]
features = np.concatenate([features, fund_feat], axis=2)
for n in range(N):
    df_tmp = pd.DataFrame(features[:, n, :])
    features[:, n, :] = df_tmp.ffill().bfill().values
clip_map = {6:(0,1),7:(-0.1,0.1),8:(-1,2),9:(0,0.5),10:(0.5,2.0),11:(0.5,2.0),12:(0.5,2.0),
            13:(-0.5,0.5),14:(-0.5,0.5),15:(0,2.0),16:(0,2.0),17:(0,10.0),19:(-0.01,0.1),20:(0,0.1),21:(-0.2,0.2)}
for idx, (lo, hi) in clip_map.items():
    features[:, :, idx] = np.clip(features[:, :, idx], lo, hi)
features = np.nan_to_num(features, nan=0.0)
INPUT_DIM = features.shape[2] - 1

# Build date index
dates_pd = pd.to_datetime([d for d in dates])
print(f"Features: {features.shape}, date range: {dates_pd[0].date()} to {dates_pd[-1].date()}")

In [ ]:
# ── Graphs + Spectral Coordinates (same as v3.0) ──
M_SPECTRAL = 16
CORR_WINDOW = 60

def mp_denoise(corr, T, N):
    eigenvalues, eigenvectors = np.linalg.eigh(corr)
    gamma = N / T
    lambda_plus = (1 + np.sqrt(gamma)) ** 2
    signal_mask = eigenvalues > lambda_plus
    n_signal = signal_mask.sum()
    if 0 < n_signal < N:
        signal_sum = eigenvalues[signal_mask].sum()
        noise_val = max((N - signal_sum) / (N - n_signal), 0.0)
        eigenvalues[~signal_mask] = noise_val
    corr_denoised = (eigenvectors * eigenvalues) @ eigenvectors.T
    d = np.sqrt(np.maximum(np.diag(corr_denoised), 1e-10))
    corr_denoised /= np.outer(d, d)
    np.fill_diagonal(corr_denoised, 1.0)
    return corr_denoised.astype(np.float32), n_signal

def spectral_coords_from_graph(A, n_dims, name=""):
    A = A + 0.001 * (1 - np.eye(A.shape[0], dtype=np.float32))
    L = laplacian(A, normed=True)
    evals, evecs = eigh(L)
    nonzero = np.where(evals > 1e-10)[0]
    if len(nonzero) >= n_dims:
        sc = evecs[:, nonzero[:n_dims]].astype(np.float32)
    else:
        sc = np.zeros((A.shape[0], n_dims), dtype=np.float32)
        if len(nonzero) > 0: sc[:, :len(nonzero)] = evecs[:, nonzero].astype(np.float32)
    sc = (sc - sc.mean(axis=0)) / (sc.std(axis=0) + 1e-8)
    return sc

def build_adjacency(labels):
    L = labels.reshape(-1, 1)
    A = ((L == L.T) & (L != -1)).astype(np.float32)
    np.fill_diagonal(A, 0)
    return A

A_sector = build_adjacency(sectors)
sc_sector = spectral_coords_from_graph(A_sector, 5, "Sector")

tnic = pd.read_csv(f"{DATA}/tnic3_filtered.csv")
gvkey_to_idx = {}
for _, row in compustat[["GVKEY", "PERMNO"]].drop_duplicates("GVKEY", keep="last").iterrows():
    try:
        gvk = int(row["GVKEY"])
        if row["PERMNO"] in permno_to_idx: gvkey_to_idx[gvk] = permno_to_idx[row["PERMNO"]]
    except: pass
tnic_year = tnic["year"].max()
tnic_sub = tnic[(tnic["year"] == tnic_year) & (tnic["gvkey1"] != tnic["gvkey2"])]
A_tnic = np.zeros((N_STOCKS, N_STOCKS), dtype=np.float32)
for _, row in tnic_sub.iterrows():
    i = gvkey_to_idx.get(int(row["gvkey1"])); j = gvkey_to_idx.get(int(row["gvkey2"]))
    if i is not None and j is not None: A_tnic[i,j] = row["score"]; A_tnic[j,i] = row["score"]
sc_tnic = spectral_coords_from_graph(A_tnic, 6, "TNIC")

own = pd.read_csv(f"{DATA}/ownership_13f.csv")
own["cusip8"] = own["cusip"].astype(str).str[:8]
cusip8_to_idx = {}
cusip_lookup = crsp_sub.drop_duplicates("PERMNO", keep="last")[["PERMNO", "HdrCUSIP"]]
for _, row in cusip_lookup.iterrows():
    if row["PERMNO"] in permno_to_idx:
        c8 = str(row["HdrCUSIP"])[:8]
        if c8 and c8 != 'nan': cusip8_to_idx[c8] = permno_to_idx[row["PERMNO"]]
own_our = own[own["cusip8"].isin(cusip8_to_idx)].copy()
own_our["stock_idx"] = own_our["cusip8"].map(cusip8_to_idx)
mgr_cats = pd.Categorical(own_our["mgrno"])
H_sparse = csr_matrix(
    (np.ones(len(own_our)), (own_our["stock_idx"].values, mgr_cats.codes)),
    shape=(N_STOCKS, len(mgr_cats.categories)))
H_binary = (H_sparse > 0).astype(np.float32)
intersection = (H_binary @ H_binary.T).toarray()
holdings_per_stock = np.array(H_binary.sum(axis=1)).flatten()
union = holdings_per_stock[:, None] + holdings_per_stock[None, :] - intersection
jaccard = np.where(union > 0, intersection / union, 0).astype(np.float32)
np.fill_diagonal(jaccard, 0)
upper_tri = jaccard[np.triu_indices(N_STOCKS, 1)]
nonzero_sim = upper_tri[upper_tri > 0]
threshold = np.quantile(nonzero_sim, 0.90) if len(nonzero_sim) > 0 else 0
A_ownership = (jaccard > threshold).astype(np.float32) if len(nonzero_sim) > 0 else np.zeros((N_STOCKS, N_STOCKS), dtype=np.float32)
sc_ownership = spectral_coords_from_graph(A_ownership, 5, "Ownership")

spectral_coords = np.concatenate([sc_sector, sc_tnic, sc_ownership], axis=1)
print(f"Spectral coords: {spectral_coords.shape}")

In [ ]:
# ── Dataset + correlation cache ──
LOOKBACK = 20; HORIZON = 5; CORR_REUSE = 5

valid_times = []
T_total = features.shape[0]
for t in range(LOOKBACK, T_total - HORIZON):
    c_now = features[t, :, 0]; c_fut = features[t + HORIZON, :, 0]
    if (~((c_now == 0) | (c_fut == 0))).sum() >= N_STOCKS * 0.5:
        valid_times.append(t)

n_total = len(valid_times)
n_train = int(0.7 * n_total); n_val = int(0.15 * n_total)
test_times = valid_times[n_train + n_val:]

# Map time indices to calendar dates
time_to_date = {t: dates_pd[t] for t in valid_times}
test_dates = [time_to_date[t] for t in test_times]
print(f"Test period: {test_dates[0].date()} to {test_dates[-1].date()} ({len(test_times)} samples)")

print("Precomputing correlation cache...")
unique_anchors = sorted(set(t // CORR_REUSE * CORR_REUSE for t in valid_times))
corr_cache = {}
for anchor in unique_anchors:
    t_start = max(0, anchor - CORR_WINDOW)
    returns = features[t_start:anchor, :, 5]
    T_window = returns.shape[0]
    if T_window >= 20:
        corr = np.corrcoef(returns.T).astype(np.float32)
        corr = np.nan_to_num(corr, nan=0.0)
        corr, _ = mp_denoise(corr, T_window, N_STOCKS)
    else:
        corr = np.eye(N_STOCKS, dtype=np.float32)
    corr_cache[anchor] = corr
print(f"Cached {len(corr_cache)} matrices")

In [ ]:
# ── Model definitions (same as v3.0) ──

class BaselineLSTM(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
    def forward(self, x):
        B, N, T, F = x.shape
        _, (h, _) = self.lstm(x.reshape(B * N, T, F))
        return self.head(h[-1]).squeeze(-1).reshape(B, N)

class WIRE(nn.Module):
    def __init__(self, d_model, m_spectral):
        super().__init__()
        omega = torch.zeros(d_model // 2, m_spectral)
        freqs = 1.0 / (10000.0 ** (2.0 * torch.arange(d_model // 2).float() / d_model))
        omega[:, 0] = freqs; omega[:, 1:] = torch.randn(d_model // 2, m_spectral - 1) * 0.01
        self.omega = nn.Parameter(omega)
    def forward(self, z, sc):
        B, N, H, d = z.shape; nb = d // 2
        theta = sc @ self.omega.T
        z = z.reshape(B, N, H, nb, 2)
        cos_t = torch.cos(theta)[None, :, None, :, None]
        sin_t = torch.sin(theta)[None, :, None, :, None]
        z0, z1 = z[..., 0:1], z[..., 1:2]
        return torch.cat([z0 * cos_t - z1 * sin_t, z0 * sin_t + z1 * cos_t], dim=-1).reshape(B, N, H, d)

class WIREAttentionLayer(nn.Module):
    def __init__(self, d_model=64, n_heads=4, m_spectral=M_SPECTRAL, dropout=0.5):
        super().__init__()
        self.d_model, self.n_heads, self.d_k = d_model, n_heads, d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model); self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model); self.W_o = nn.Linear(d_model, d_model)
        self.wire = WIRE(self.d_k, m_spectral)
        self.dropout = nn.Dropout(dropout); self.norm = nn.LayerNorm(d_model)
        self.edge_bias = nn.Parameter(torch.full((n_heads,), 0.1))
    def forward(self, x, sc, corr=None):
        B, N, _ = x.shape; H, dk = self.n_heads, self.d_k
        Q = self.wire(self.W_q(x).reshape(B, N, H, dk), sc)
        K = self.wire(self.W_k(x).reshape(B, N, H, dk), sc)
        V = self.W_v(x).reshape(B, N, H, dk)
        Q, K, V = [t.permute(0, 2, 1, 3) for t in (Q, K, V)]
        attn_scores = Q @ K.transpose(-2, -1) / dk ** 0.5
        if corr is not None: attn_scores = attn_scores + self.edge_bias[None, :, None, None] * corr[:, None, :, :]
        attn = self.dropout(F.softmax(attn_scores, dim=-1))
        out = self.W_o((attn @ V).permute(0, 2, 1, 3).reshape(B, N, self.d_model))
        return self.norm(x + out), attn

class GraphTransformerWIRE(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dim=64, n_heads=4, n_layers=2, m_spectral=M_SPECTRAL, dropout=0.5):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.attn_layers = nn.ModuleList([WIREAttentionLayer(hidden_dim, n_heads, m_spectral, dropout) for _ in range(n_layers)])
        self.ffns = nn.ModuleList([nn.Sequential(nn.Linear(hidden_dim, hidden_dim*2), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_dim*2, hidden_dim), nn.LayerNorm(hidden_dim)) for _ in range(n_layers)])
        self.head_lstm = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.head_wire = nn.Sequential(nn.Linear(hidden_dim, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        self.mix = nn.Parameter(torch.tensor(0.0))
    def forward(self, x, sc, corr=None):
        B, N, T, F = x.shape
        _, (h, _) = self.lstm(x.reshape(B * N, T, F))
        embed = h[-1].reshape(B, N, self.hidden_dim)
        pred_lstm = self.head_lstm(embed).squeeze(-1)
        z = embed
        attn_maps = []
        for al, ff in zip(self.attn_layers, self.ffns):
            z, attn = al(z, sc, corr); z = ff(z) + z; attn_maps.append(attn)
        pred_wire = self.head_wire(z).squeeze(-1)
        alpha = torch.sigmoid(self.mix)
        return alpha * pred_lstm + (1 - alpha) * pred_wire, attn_maps

print("Model classes defined.")

In [ ]:
# ── Load saved models ──
# NOTE: v3.0 must save models after training. Add this to v3.0's output cell:
#   torch.save(best_wire_state, f"{OUT_DIR}/best_wire.pt")
#   torch.save(best_lstm_state, f"{OUT_DIR}/best_lstm.pt")

wire_path = f"{V3_DIR}/best_wire.pt"
lstm_path = f"{V3_DIR}/best_lstm.pt"

if os.path.exists(wire_path) and os.path.exists(lstm_path):
    model_wire = GraphTransformerWIRE().to(device)
    model_wire.load_state_dict(torch.load(wire_path, map_location=device))
    model_wire.eval()
    model_lstm = BaselineLSTM().to(device)
    model_lstm.load_state_dict(torch.load(lstm_path, map_location=device))
    model_lstm.eval()
    print("Models loaded successfully.")
    MODELS_AVAILABLE = True
else:
    print("WARNING: Saved models not found.")
    print(f"  Expected: {wire_path}")
    print(f"  Expected: {lstm_path}")
    print("  Add these lines to v3.0's last code cell after training:")
    print('    torch.save(best_wire_state, f"{OUT_DIR}/best_wire.pt")')
    print('    torch.save(best_lstm_state, f"{OUT_DIR}/best_lstm.pt")')
    MODELS_AVAILABLE = False

sc_tensor = torch.tensor(spectral_coords, dtype=torch.float32).to(device)

## 1. Fama-French 3-Factor Alpha

We form a long-short portfolio from model predictions:
- **Long:** top decile (30 stocks) by predicted cross-sectional rank
- **Short:** bottom decile (30 stocks)
- **Portfolio return:** equal-weighted long minus short, daily

Then regress on Fama-French 3 factors:

$$R_{LS,t} - R_{f,t} = \alpha + \beta \cdot MktRF_t + s \cdot SMB_t + h \cdot HML_t + \epsilon_t$$

Significant positive alpha = the model captures signals beyond size/value/market.

In [ ]:
# Download Fama-French 3-factor daily data
import urllib.request, zipfile

FF_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_daily_CSV.zip"
ff_zip_path = "/tmp/ff3_daily.zip"
ff_csv_path = "/tmp/ff3_daily.csv"

print("Downloading Fama-French 3-factor daily data...")
urllib.request.urlretrieve(FF_URL, ff_zip_path)

with zipfile.ZipFile(ff_zip_path, 'r') as z:
    csv_name = [f for f in z.namelist() if f.endswith('.CSV') or f.endswith('.csv')][0]
    with z.open(csv_name) as f:
        content = f.read().decode('utf-8')

# Parse: skip header rows, read until annual section
lines = content.strip().split('\n')
start_idx = None
for i, line in enumerate(lines):
    if line.strip().startswith('19') or line.strip().startswith('20'):
        start_idx = i
        break

data_lines = []
for line in lines[start_idx:]:
    parts = line.strip().split(',')
    if len(parts) >= 5 and len(parts[0].strip()) == 8:  # YYYYMMDD format
        data_lines.append(parts)
    elif len(parts) < 5 or not parts[0].strip():
        break  # End of daily section

ff_df = pd.DataFrame(data_lines, columns=["date", "Mkt-RF", "SMB", "HML", "RF"])
ff_df["date"] = pd.to_datetime(ff_df["date"].str.strip())
for col in ["Mkt-RF", "SMB", "HML", "RF"]:
    ff_df[col] = pd.to_numeric(ff_df[col].str.strip(), errors='coerce') / 100  # Convert from % to decimal
ff_df = ff_df.dropna()
print(f"FF3 factors: {len(ff_df)} daily observations, {ff_df['date'].min().date()} to {ff_df['date'].max().date()}")

In [ ]:
assert MODELS_AVAILABLE, "Need saved models to compute Fama-French alpha."

# Generate daily predictions on test set
print("Generating test set predictions...")

wire_preds_all = []
lstm_preds_all = []
actual_returns_all = []
pred_dates = []

model_wire.eval(); model_lstm.eval()
with torch.no_grad():
    for t_idx, t in enumerate(test_times):
        x = features[t - LOOKBACK:t, :, 1:]
        x = np.nan_to_num(x, nan=0.0)
        x_t = torch.from_numpy(x.transpose(1, 0, 2).copy()).unsqueeze(0).to(device)  # (1, N, T, F)
        anchor = t // CORR_REUSE * CORR_REUSE
        corr = torch.from_numpy(corr_cache[anchor]).unsqueeze(0).to(device)  # (1, N, N)

        wire_pred, _ = model_wire(x_t, sc_tensor, corr)
        lstm_pred = model_lstm(x_t)

        wire_preds_all.append(wire_pred[0].cpu().numpy())
        lstm_preds_all.append(lstm_pred[0].cpu().numpy())

        # Actual forward returns (raw, not rank-normalized)
        c_now = features[t, :, 0]
        c_fut = features[t + HORIZON, :, 0]
        actual_ret = np.where(c_now != 0, c_fut / c_now - 1, 0)
        actual_returns_all.append(actual_ret)
        pred_dates.append(time_to_date[t])

wire_preds = np.array(wire_preds_all)  # (n_test, N)
lstm_preds = np.array(lstm_preds_all)
actual_rets = np.array(actual_returns_all)
pred_dates = pd.DatetimeIndex(pred_dates)
print(f"Predictions: {wire_preds.shape}, dates: {pred_dates[0].date()} to {pred_dates[-1].date()}")

In [ ]:
# Form long-short portfolios
DECILE = N_STOCKS // 10  # top/bottom 30 stocks

def long_short_returns(predictions, actual_returns):
    """Equal-weighted long top decile, short bottom decile."""
    n_days = predictions.shape[0]
    ls_returns = np.zeros(n_days)
    for day in range(n_days):
        ranks = np.argsort(np.argsort(-predictions[day]))  # 0 = highest predicted
        long_mask = ranks < DECILE
        short_mask = ranks >= (N_STOCKS - DECILE)
        ls_returns[day] = actual_returns[day, long_mask].mean() - actual_returns[day, short_mask].mean()
    return ls_returns

wire_ls = long_short_returns(wire_preds, actual_rets)
lstm_ls = long_short_returns(lstm_preds, actual_rets)

print(f"WIRE L/S: mean={wire_ls.mean()*252:.4f} (annualized), Sharpe={wire_ls.mean()/wire_ls.std()*np.sqrt(252):.2f}")
print(f"LSTM L/S: mean={lstm_ls.mean()*252:.4f} (annualized), Sharpe={lstm_ls.mean()/lstm_ls.std()*np.sqrt(252):.2f}")

In [ ]:
# Fama-French 3-factor regression
import statsmodels.api as sm

# Merge portfolio returns with FF factors by date
port_df = pd.DataFrame({"date": pred_dates, "wire_ls": wire_ls, "lstm_ls": lstm_ls})
port_df = port_df.merge(ff_df, on="date", how="inner")
print(f"Matched {len(port_df)} trading days with FF factors")

def run_ff3_regression(returns, port_df, name):
    """Run Fama-French 3-factor regression and print results."""
    y = returns - port_df["RF"].values  # Excess returns
    X = sm.add_constant(port_df[["Mkt-RF", "SMB", "HML"]].values)
    model = sm.OLS(y, X).fit(cov_type='HC1')  # Heteroscedasticity-robust SEs

    print(f"\n{'=' * 60}")
    print(f"  Fama-French 3-Factor: {name}")
    print(f"{'=' * 60}")
    print(f"  Alpha (daily):      {model.params[0]:+.6f}  (t={model.tvalues[0]:.2f}, p={model.pvalues[0]:.3f})")
    print(f"  Alpha (annualized): {model.params[0]*252:+.4f}")
    print(f"  Mkt-RF beta:        {model.params[1]:+.4f}  (t={model.tvalues[1]:.2f})")
    print(f"  SMB loading:        {model.params[2]:+.4f}  (t={model.tvalues[2]:.2f})")
    print(f"  HML loading:        {model.params[3]:+.4f}  (t={model.tvalues[3]:.2f})")
    print(f"  R-squared:          {model.rsquared:.4f}")
    sig = "SIGNIFICANT" if model.pvalues[0] < 0.05 else "not significant"
    print(f"  Alpha is {sig} at 5% level.")
    return model

model_wire_ff = run_ff3_regression(port_df["wire_ls"].values, port_df, "WIRE Long-Short")
model_lstm_ff = run_ff3_regression(port_df["lstm_ls"].values, port_df, "LSTM Long-Short")

In [ ]:
# Fama-French visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Cumulative L/S returns
axes[0].plot(port_df["date"], np.cumsum(port_df["wire_ls"]), label="WIRE L/S", lw=2)
axes[0].plot(port_df["date"], np.cumsum(port_df["lstm_ls"]), label="LSTM L/S", lw=2, alpha=0.7)
axes[0].axhline(0, color="gray", ls=":", alpha=0.5)
axes[0].set_xlabel("Date"); axes[0].set_ylabel("Cumulative Return")
axes[0].set_title("Long-Short Portfolio Cumulative Returns")
axes[0].legend()

# Alpha comparison
alphas = [model_lstm_ff.params[0] * 252, model_wire_ff.params[0] * 252]
alpha_se = [model_lstm_ff.bse[0] * 252 * 1.96, model_wire_ff.bse[0] * 252 * 1.96]
axes[1].bar(["LSTM", "WIRE"], alphas, yerr=alpha_se, capsize=10,
            color=["#1f77b4", "#ff7f0e"], alpha=0.8, edgecolor="black")
axes[1].axhline(0, color="gray", ls=":", alpha=0.5)
axes[1].set_ylabel("Annualized Alpha")
axes[1].set_title("FF3 Alpha (with 95% CI)")

# Rolling Sharpe (60-day)
roll = 60
wire_rolling_sharpe = pd.Series(wire_ls).rolling(roll).apply(
    lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0)
lstm_rolling_sharpe = pd.Series(lstm_ls).rolling(roll).apply(
    lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0)
axes[2].plot(pred_dates[:len(wire_rolling_sharpe)], wire_rolling_sharpe, label="WIRE", lw=1.5)
axes[2].plot(pred_dates[:len(lstm_rolling_sharpe)], lstm_rolling_sharpe, label="LSTM", lw=1.5, alpha=0.7)
axes[2].axhline(0, color="gray", ls=":", alpha=0.5)
axes[2].set_xlabel("Date"); axes[2].set_ylabel("Rolling Sharpe (60-day)")
axes[2].set_title("Rolling Sharpe Ratio")
axes[2].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/fama_french_alpha.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Shock Event Attention Analysis

We extract WIRE attention maps during market stress periods and compare to normal periods:
- **COVID crash:** Feb 20 - Mar 23, 2020 (S&P 500 fell ~34%)
- **2022 rate hikes:** Jan - Jun 2022 (Fed tightening cycle)
- **Normal:** remaining test period

Hypothesis: during shocks, attention should concentrate more within sectors (risk contagion) and correlation-driven edges should strengthen.

In [ ]:
assert MODELS_AVAILABLE, "Need saved models for attention analysis."

# Define shock event windows
SHOCK_EVENTS = {
    "COVID crash": (pd.Timestamp("2020-02-20"), pd.Timestamp("2020-03-23")),
    "2022 rate hikes": (pd.Timestamp("2022-01-03"), pd.Timestamp("2022-06-15")),
}

# Classify test samples into shock vs normal
shock_indices = {name: [] for name in SHOCK_EVENTS}
normal_indices = []

for i, t in enumerate(test_times):
    d = time_to_date[t]
    in_shock = False
    for name, (start, end) in SHOCK_EVENTS.items():
        if start <= d <= end:
            shock_indices[name].append(i)
            in_shock = True
            break
    if not in_shock:
        normal_indices.append(i)

for name, indices in shock_indices.items():
    if indices:
        print(f"{name}: {len(indices)} test samples ({time_to_date[test_times[indices[0]]].date()} to {time_to_date[test_times[indices[-1]]].date()})")
    else:
        print(f"{name}: NOT in test period")
print(f"Normal: {len(normal_indices)} test samples")

In [ ]:
# Extract attention maps for shock and normal periods
def extract_attention_stats(indices, max_samples=100):
    """Extract attention statistics for given test indices."""
    if not indices:
        return None
    sample_idx = indices[:max_samples]  # Cap for memory
    sector_ratios = []
    entropy_values = []
    attention_maps = []

    model_wire.eval()
    with torch.no_grad():
        for i in sample_idx:
            t = test_times[i]
            x = features[t - LOOKBACK:t, :, 1:]
            x = np.nan_to_num(x, nan=0.0)
            x_t = torch.from_numpy(x.transpose(1, 0, 2).copy()).unsqueeze(0).to(device)
            anchor = t // CORR_REUSE * CORR_REUSE
            corr = torch.from_numpy(corr_cache[anchor]).unsqueeze(0).to(device)

            _, attn_maps = model_wire(x_t, sc_tensor, corr)
            attn = attn_maps[-1][0].cpu().numpy()  # Last layer, all heads: (H, N, N)

            # Per-head sector attention ratio
            for h in range(attn.shape[0]):
                a = attn[h]
                same = a[A_sector > 0].mean()
                diff = a[(A_sector == 0) & ~np.eye(N_STOCKS, dtype=bool)].mean()
                sector_ratios.append(same / diff if diff > 0 else 1.0)

            # Attention entropy (head-averaged)
            a_avg = attn.mean(axis=0)  # Average across heads
            for row in a_avg:
                row_norm = row / (row.sum() + 1e-10)
                ent = -np.sum(row_norm * np.log(row_norm + 1e-10))
                entropy_values.append(ent)

            if len(attention_maps) < 3:  # Save a few for visualization
                attention_maps.append(attn.mean(axis=0))

    return {
        "sector_ratio_mean": np.mean(sector_ratios),
        "sector_ratio_std": np.std(sector_ratios),
        "entropy_mean": np.mean(entropy_values),
        "entropy_std": np.std(entropy_values),
        "attention_maps": attention_maps,
        "n_samples": len(sample_idx),
    }

normal_stats = extract_attention_stats(normal_indices)
shock_stats = {}
for name, indices in shock_indices.items():
    if indices:
        shock_stats[name] = extract_attention_stats(indices)

print(f"\n{'=' * 65}")
print(f"  ATTENTION STATISTICS: Shock vs Normal")
print(f"{'=' * 65}")
print(f"{'Period':<25} {'Sector Ratio':>15} {'Attention Entropy':>20} {'N':>5}")
print("-" * 70)
print(f"{'Normal':<25} {normal_stats['sector_ratio_mean']:>12.3f}+/-{normal_stats['sector_ratio_std']:.3f}  "
      f"{normal_stats['entropy_mean']:>15.3f}+/-{normal_stats['entropy_std']:.3f} {normal_stats['n_samples']:>5}")
for name, stats in shock_stats.items():
    print(f"{name:<25} {stats['sector_ratio_mean']:>12.3f}+/-{stats['sector_ratio_std']:.3f}  "
          f"{stats['entropy_mean']:>15.3f}+/-{stats['entropy_std']:.3f} {stats['n_samples']:>5}")

In [ ]:
# Attention visualization: shock vs normal
n_shock_plots = sum(1 for s in shock_stats.values() if s and s['attention_maps'])
n_plots = 1 + n_shock_plots  # normal + each shock period
fig, axes = plt.subplots(1, max(n_plots, 2), figsize=(7 * max(n_plots, 2), 6))
if n_plots == 1: axes = [axes]

# Sort by sector for visualization
sector_order = np.argsort(sectors)

# Normal period
if normal_stats and normal_stats['attention_maps']:
    a = normal_stats['attention_maps'][0]
    sns.heatmap(a[np.ix_(sector_order, sector_order)], cmap="viridis",
                xticklabels=False, yticklabels=False, ax=axes[0])
    axes[0].set_title(f"Normal (ratio={normal_stats['sector_ratio_mean']:.2f}x, H={normal_stats['entropy_mean']:.1f})")

# Shock periods
ax_idx = 1
for name, stats in shock_stats.items():
    if stats and stats['attention_maps'] and ax_idx < len(axes):
        a = stats['attention_maps'][0]
        sns.heatmap(a[np.ix_(sector_order, sector_order)], cmap="viridis",
                    xticklabels=False, yticklabels=False, ax=axes[ax_idx])
        axes[ax_idx].set_title(f"{name} (ratio={stats['sector_ratio_mean']:.2f}x, H={stats['entropy_mean']:.1f})")
        ax_idx += 1

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/shock_attention.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Time series of attention concentration
print("Computing rolling attention statistics...")

# Sample every 5th test day for efficiency
sample_rate = 5
daily_sector_ratio = []
daily_entropy = []
daily_dates = []

model_wire.eval()
with torch.no_grad():
    for i in range(0, len(test_times), sample_rate):
        t = test_times[i]
        x = features[t - LOOKBACK:t, :, 1:]
        x = np.nan_to_num(x, nan=0.0)
        x_t = torch.from_numpy(x.transpose(1, 0, 2).copy()).unsqueeze(0).to(device)
        anchor = t // CORR_REUSE * CORR_REUSE
        corr = torch.from_numpy(corr_cache[anchor]).unsqueeze(0).to(device)

        _, attn_maps = model_wire(x_t, sc_tensor, corr)
        a = attn_maps[-1][0].mean(dim=0).cpu().numpy()  # Head-averaged

        same = a[A_sector > 0].mean()
        diff = a[(A_sector == 0) & ~np.eye(N_STOCKS, dtype=bool)].mean()
        daily_sector_ratio.append(same / diff if diff > 0 else 1.0)

        ent = -np.sum(a / a.sum(axis=1, keepdims=True) * np.log(a / a.sum(axis=1, keepdims=True) + 1e-10), axis=1).mean()
        daily_entropy.append(ent)
        daily_dates.append(time_to_date[t])

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(daily_dates, daily_sector_ratio, lw=1, alpha=0.6)
axes[0].plot(daily_dates, pd.Series(daily_sector_ratio).rolling(20, min_periods=1).mean(), lw=2, label="20-day MA")
for name, (start, end) in SHOCK_EVENTS.items():
    axes[0].axvspan(start, end, color="red", alpha=0.15, label=name)
axes[0].axhline(1.0, color="gray", ls="--", alpha=0.5)
axes[0].set_ylabel("Sector Attention Ratio"); axes[0].set_title("Sector Attention Concentration Over Time")
axes[0].legend(loc="upper left", fontsize=8)

axes[1].plot(daily_dates, daily_entropy, lw=1, alpha=0.6)
axes[1].plot(daily_dates, pd.Series(daily_entropy).rolling(20, min_periods=1).mean(), lw=2, label="20-day MA")
for name, (start, end) in SHOCK_EVENTS.items():
    axes[1].axvspan(start, end, color="red", alpha=0.15, label=name)
axes[1].set_ylabel("Attention Entropy"); axes[1].set_xlabel("Date")
axes[1].set_title("Attention Entropy Over Time (lower = more concentrated)")
axes[1].legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/attention_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Failure Mode Documentation

### 3.1 Sparse Graph Spectral Instability

**Problem:** TNIC competitor and 13F ownership graphs have many disconnected components (52 and 40 respectively) with dozens of isolated nodes. The normalized Laplacian eigendecomposition assigns degenerate (zero) spectral coordinates to isolated nodes, making WIRE unable to differentiate their positions.

**Root cause:** 
- TNIC graph covers only ~75% of our 300-stock universe (GVKEY mapping gaps)
- 13F ownership requires Jaccard similarity > 90th percentile threshold, leaving many stocks disconnected
- Spectral gap = 0 means the Fiedler vector is degenerate

**Mitigation:** Connectivity perturbation `A + 0.001 * (1 - I)` — analogous to PageRank teleportation. Adds infinitesimal uniform edges to make the graph fully connected while preserving original structure at 1000:1 ratio for binary edges.

**Impact:** Without perturbation, WIRE IC was -0.009 (worse than random). With perturbation, spectral gap becomes positive and spectral coordinates become meaningful.

### 3.2 Supply Chain Data Unusability

**Problem:** The Compustat Segments supply chain dataset (`supplychain_segments.csv`) reports customer company names as free text (e.g., "WALMART STORES INC", "WAL MART", "US GOVERNMENT"). Matching these to PERMNO/GVKEY identifiers is unreliable.

**Attempts:**
- Ticker matching: ~50 edges out of 300 stocks — too sparse for meaningful graph
- Fuzzy string matching: high false positive rate (e.g., "APPLE" matches "APPLE HOSPITALITY REIT")
- CUSIP/GVKEY from segment data: not available in the customer name field

**Decision:** Dropped supply chain graph. Replaced with 13F ownership co-holding overlap (Jaccard similarity), which provides a comparable economic relationship signal with clean identifiers.

### 3.3 Long Training Window (T=1000) Failure

**Problem:** TA suggested using T=1000 day correlation window. WIRE lost 4/5 seeds (mean IC +0.036 vs LSTM +0.047).

**Root cause:** With CORR_REUSE=20, T=1000 correlation matrices become nearly static — the 20-day reuse window sees almost no change in a 1000-day rolling window. This eliminates the dynamic market state signal that WIRE's edge_bias mechanism depends on.

**Lesson:** T=60 dynamic correlation is the sweet spot. The correlation graph serves as a *complement* to fundamental spectral coordinates, capturing short-term regime changes. Making it static removes this complementary signal.

In [ ]:
# Save all results
results = {
    "fama_french": {
        "wire": {
            "alpha_daily": float(model_wire_ff.params[0]),
            "alpha_annual": float(model_wire_ff.params[0] * 252),
            "alpha_tstat": float(model_wire_ff.tvalues[0]),
            "alpha_pval": float(model_wire_ff.pvalues[0]),
            "mkt_beta": float(model_wire_ff.params[1]),
            "smb": float(model_wire_ff.params[2]),
            "hml": float(model_wire_ff.params[3]),
            "r_squared": float(model_wire_ff.rsquared),
        },
        "lstm": {
            "alpha_daily": float(model_lstm_ff.params[0]),
            "alpha_annual": float(model_lstm_ff.params[0] * 252),
            "alpha_tstat": float(model_lstm_ff.tvalues[0]),
            "alpha_pval": float(model_lstm_ff.pvalues[0]),
            "mkt_beta": float(model_lstm_ff.params[1]),
            "smb": float(model_lstm_ff.params[2]),
            "hml": float(model_lstm_ff.params[3]),
            "r_squared": float(model_lstm_ff.rsquared),
        },
    },
    "shock_events": {
        "normal": {"sector_ratio": normal_stats["sector_ratio_mean"], "entropy": normal_stats["entropy_mean"]},
        **{name: {"sector_ratio": s["sector_ratio_mean"], "entropy": s["entropy_mean"]} for name, s in shock_stats.items()},
    },
}

with open(f"{OUT_DIR}/post_analysis.json", "w") as f:
    json.dump(results, f, indent=2, default=str)
print(f"Saved to {OUT_DIR}/post_analysis.json")